# 🥈 Silver Layer: Cleaned & Validated

### 🎯 Objective
Inspect the transformed data and demonstrate the **Write-Audit-Publish (WAP)** pattern.

### 🌿 Branch Strategy: `silver` -> `main`
- **Purpose**: High-quality, trustworthy data.
- **WAP Pattern**:
  1.  **Write**: We created a `silver` branch off `main`.
  2.  **Audit**: We ran `audit_silver_quality.py` to check for nulls and duplicates.
  3.  **Publish**: We merged `silver` into `main` ONLY after passing checks.
- **Nessie Reference**: `nessie.ecommerce.orders_silver@main` (The published version)

In [ ]:
import os
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("Silver_Demo") \
    .config("spark.jars.packages", "org.apache.iceberg:iceberg-spark-runtime-3.3_2.12:1.3.1,org.projectnessie.nessie-integrations:nessie-spark-extensions-3.3_2.12:0.67.0,software.amazon.awssdk:bundle:2.17.178,software.amazon.awssdk:url-connection-client:2.17.178") \
    .config("spark.sql.extensions", "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions,org.projectnessie.spark.extensions.NessieSparkSessionExtensions") \
    .config("spark.sql.catalog.nessie", "org.apache.iceberg.spark.SparkCatalog") \
    .config("spark.sql.catalog.nessie.catalog-impl", "org.apache.iceberg.nessie.NessieCatalog") \
    .config("spark.sql.catalog.nessie.uri", "http://nessie:19120/api/v1") \
    .config("spark.sql.catalog.nessie.ref", "main") \
    .config("spark.sql.catalog.nessie.authentication.type", "NONE") \
    .config("spark.sql.catalog.nessie.warehouse", "s3a://lakehouse/warehouse") \
    .config("spark.sql.catalog.nessie.io-impl", "org.apache.iceberg.aws.s3.S3FileIO") \
    .config("spark.sql.catalog.nessie.s3.endpoint", "http://minio:9000") \
    .config("spark.hadoop.fs.s3a.endpoint", "http://minio:9000") \
    .config("spark.hadoop.fs.s3a.access.key", "admin") \
    .config("spark.hadoop.fs.s3a.secret.key", "password123") \
    .config("spark.hadoop.fs.s3a.path.style.access", "true") \
    .config("spark.hadoop.fs.s3a.connection.ssl.enabled", "false") \
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem") \
    .getOrCreate()

print("✅ Spark Session Connected to Main Branch (Production)")

### 🧹 Transformation Evidence
Notice the new columns: `order_date` (parsed), `processed_at`, and `data_quality_score`.

In [ ]:
# Read from the production MAIN branch
df_silver = spark.sql("SELECT customer_id, event_type, price, order_date, processed_at FROM nessie.ecommerce.`orders_silver@main` LIMIT 10")
df_silver.show(truncate=False)

### 🚀 Hidden Partitioning in Action
We partitioned by `days(order_date)`. Let's see how Spark pushes down this filter.
This query should be extremely fast because it skips files not in '2020-01-01'.

In [ ]:
spark.sql("SELECT * FROM nessie.ecommerce.`orders_silver@main` WHERE order_date = '2020-01-01'").explain()